# IQ on human - K4me3 vs K27me3

In [2]:
library(prego)
library(iceqream)
options(tidyverse.quiet = TRUE)
library(tidyverse)
library(purrr)
library(tgutil)
library(misha)
library(misha.ext)
library(glue)
library(here)
library(tgstat)

gsetroot(here("data/hg19"))
options(gmax.data.size = 1e+9)
source(here("code/model-utils.R"))

Loading required package: misha

Warning message in system("timedatectl", intern = TRUE):
“running command 'timedatectl' had status 1”
here() starts at /net/mraid20/ifs/wisdom/tanay_lab/tgdata/users/evghenic/Proj/polycomb/seq2epi_paper_raw



In [3]:
setwd(here())

In [ ]:
#iq-hg19-5mc-vs-nothing-model_preds.tsv

In [64]:
#cgd_all <- fread(here("output/iq-hg19-5mc-vs-nothing-model_preds.tsv")) %>% arrange(chrom,start)

In [5]:
setwd(here())

In [6]:
cgd_all = fread( here("output/cgd_hg19_iq.tsv"), sep = "\t", quote = FALSE)

In [7]:
head(cgd_all)

,chrom,start,end,id,l,es_k27_max,es_k4_max,es_k27_cov,es_k4_cov,es_biv_cov,⋯,l10_h1_k4_sum,l10_h1_k27_sum,pcg,txg,l500,f_out,txg_pcg,resp10,f_ambig,tss_gene
,<chr>,<int>,<int>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<lgl>,<lgl>,<lgl>,<lgl>,<dbl>,<int>,<lgl>,<chr>
1,chr1,9488701,9489371,5S_rRNA.1,670,5.00,25.60,0.00000000,0.0000000,0.00000000,⋯,5.104462,3.665230,FALSE,FALSE,TRUE,TRUE,NA,NA,FALSE,5S_rRNA
2,chr1,41826605,41828380,5S_rRNA.2,1775,110.25,90.60,0.38202247,0.0000000,0.33707865,⋯,6.128760,6.101676,FALSE,FALSE,TRUE,FALSE,0.1035574,NA,TRUE,5S_rRNA
3,chr1,41831862,41832691,5S_rRNA.3,829,17.00,88.75,0.00000000,0.7619048,0.00000000,⋯,6.706934,4.651147,FALSE,FALSE,TRUE,FALSE,0.4733907,NA,TRUE,5S_rRNA
4,chr1,41847183,41849391,5S_rRNA.4,2208,29.45,117.05,0.05405405,0.8018018,0.01801802,⋯,6.573153,4.947160,FALSE,FALSE,TRUE,FALSE,0.3946410,NA,TRUE,5S_rRNA
5,chr1,41889237,41889828,5S_rRNA.5,591,0.00,2.00,0.00000000,0.0000000,0.00000000,⋯,3.461428,3.321928,FALSE,FALSE,TRUE,TRUE,NA,NA,FALSE,5S_rRNA
6,chr1,117487141,117487793,5S_rRNA.6,652,2.00,0.00,0.00000000,0.0000000,0.00000000,⋯,3.321928,3.508475,FALSE,FALSE,TRUE,TRUE,NA,NA,FALSE,5S_rRNA


In [8]:
dim(cgd_all)

[1] 34153    45

In [9]:
dim(cgd_all)

[1] 34153    45

In [10]:
test_chroms <- c("chr2", "chr8", "chr12", "chr18")
cgd_all <- cgd_all %>% mutate(orig_resp10 = resp10) %>% 
    mutate(type = case_when(
        l500 & !f_out & chrom %in% test_chroms ~ "test",
        l500 & !f_out & !chrom %in% test_chroms ~ "train",
        TRUE ~ "other"
    )) %>% 
    mutate(resp10 = txg_pcg)
cgd_all <- cgd_all %>%    
    arrange(tss_gene, chrom, start, end) %>%
    group_by(tss_gene) %>%
    mutate(id = paste0(tss_gene, ".", row_number())) %>%
    mutate(n = n()) %>%
    mutate(id = ifelse(n == 1, tss_gene, id)) %>%
    select(-n) %>%
    ungroup() %>%
    select(chrom:end, id, everything())  %>% 
    as.data.frame() 
rownames(cgd_all) <- cgd_all$id

In [11]:
cgd_all %>%
    count(type)

type,n
<chr>,<int>
other,20151
test,2470
train,11532


In [12]:
pred_threshold <- 0.25
pROC::coords(pROC::roc(cgd_all$resp5mc, cgd_all$pred), pred_threshold, ret = c("threshold", "sensitivity", "specificity"))

ERROR: Error in roc.default(cgd_all$resp5mc, cgd_all$pred): No valid data provided.


In [13]:
pwm5 <- readr::read_rds(here("output/pwm5_norm_hg19_iq.rds"))
pwm5_pad <- readr::read_rds(here("output/pwm5_pad_norm_hg19_iq.rds"))
pwm3 <- readr::read_rds(here("output/pwm3_norm_hg19_iq.rds"))
pwm3_pad <- readr::read_rds(here("output/pwm3_pad_norm_hg19_iq.rds"))
pwm_in <- readr::read_rds(here("output/pwm_in_norm_hg19_iq.rds"))
add_feats <- readr::read_rds(here("output/add_feats_hg19_iq.rds"))
add_feats_in <- readr::read_rds(here("output/add_feats_in_hg19_iq.rds"))

In [21]:
dim(cgd_all)

[1] 34153    45

In [14]:
f <- cgd_all$type %in% c("train", "test") & cgd_all$id %in% rownames(pwm5)  & cgd_all$id %in% rownames(pwm5_pad) & cgd_all$id %in% rownames(pwm3) & cgd_all$id %in% rownames(pwm3_pad) & cgd_all$id %in% rownames(add_feats)
sum(f)
cgd <- cgd_all[f, ]
pwm5 <- pwm5[cgd$id, ]
pwm5_pad <- pwm5_pad[cgd$id, ]
pwm3 <- pwm3[cgd$id, ]
pwm3_pad <- pwm3_pad[cgd$id, ]
add_feats <- add_feats[cgd$id, ]
add_feats_in <- add_feats_in[cgd$id, ]
pwm_in <- pwm_in[cgd$id, ]

[1] 14001

In [15]:
add_feats5 <- add_feats
colnames(add_feats5) <- gsub("_5$", "", colnames(add_feats5))
add_feats3 <- add_feats
colnames(add_feats3) <- gsub("_3$", "", colnames(add_feats3))
add_feats5_pad <- add_feats
colnames(add_feats5_pad) <- gsub("_5pad$", "", colnames(add_feats5_pad))
add_feats3_pad <- add_feats
colnames(add_feats3_pad) <- gsub("_3pad$", "", colnames(add_feats3_pad))

cgd5 <- cgd[, 1:4]
cgd5$start <- ifelse(cgd$tss_strand == 1, cgd$start, cgd$end - 600)
cgd5$end <- ifelse(cgd$tss_strand == 1, cgd$start + 600, cgd$end)

cgd5_pad <- cgd[, 1:4]
cgd5_pad$start <- ifelse(cgd$tss_strand == 1, cgd$start - 500, cgd$end)
cgd5_pad$end <- ifelse(cgd$tss_strand == 1, cgd$start, cgd$end + 500)

cgd3 <- cgd[, 1:4]
cgd3$start <- ifelse(cgd$tss_strand == 1, cgd$end - 600, cgd$start)
cgd3$end <- ifelse(cgd$tss_strand == 1, cgd$end, cgd$start + 600)

cgd3_pad <- cgd[, 1:4]
cgd3_pad$start <- ifelse(cgd$tss_strand == 1, cgd$end, cgd$start - 500)
cgd3_pad$end <- ifelse(cgd$tss_strand == 1, cgd$end + 500, cgd$start)

train_idxs <- which(cgd$type == "train")
test_idxs <- which(cgd$type == "test")

cgd5_oriented <- cgd5
cgd5_oriented$strand <- cgd$tss_strand
cgd5_oriented <- cgd5_oriented %>% select(chrom, start, end, strand, everything())

cgd5_pad_oriented <- cgd5_pad
cgd5_pad_oriented$strand <- cgd$tss_strand
cgd5_pad_oriented <- cgd5_pad_oriented %>% select(chrom, start, end, strand, everything())

cgd3_oriented <- cgd3
cgd3_oriented$strand <- cgd$tss_strand
cgd3_oriented <- cgd3_oriented %>% select(chrom, start, end, strand, everything())

cgd3_pad_oriented <- cgd3_pad
cgd3_pad_oriented$strand <- cgd$tss_strand
cgd3_pad_oriented <- cgd3_pad_oriented %>% select(chrom, start, end, strand, everything())

cgd_oriented <- cgd
cgd_oriented$strand <- cgd$tss_strand
cgd_oriented <- cgd_oriented %>% select(chrom, start, end, strand, everything())

fwrite(cgd_oriented, here("output/cgd_oriented_hg19_iq_second.tsv"), sep = "\t")

In [16]:
scenic_motifs <- grep("SCENIC\\.", colnames(pwm5), value = TRUE)
pwm5 <- pwm5[, !(colnames(pwm5) %in% scenic_motifs)]
pwm5_pad <- pwm5_pad[, !(colnames(pwm5_pad) %in% scenic_motifs)]
pwm3 <- pwm3[, !(colnames(pwm3) %in% scenic_motifs)]
pwm3_pad <- pwm3_pad[, !(colnames(pwm3_pad) %in% scenic_motifs)]

In [17]:
stopifnot(all(cgd$id == rownames(pwm5)))
stopifnot(all(cgd$id == rownames(pwm5_pad)))
stopifnot(all(cgd$id == rownames(pwm3)))
stopifnot(all(cgd$id == rownames(pwm3_pad)))
stopifnot(all(cgd$id == rownames(add_feats)))
stopifnot(all(cgd$id == rownames(add_feats_in)))

In [19]:
fwrite(cgd, here("output/cgd_hg19_iq_second.tsv"), sep = "\t")

## Run models

In [20]:
#new
suffix <- "_prego_10_motifs_no_add_oriented_hg19_pcg_vs_txg"
run_iq_regression(
    model_type = "5",
    cgd = cgd,
    test_idxs = test_idxs,
    train_idxs = train_idxs,
    suffix = suffix
)
run_iq_regression(
    model_type = "3",
    cgd = cgd,
    test_idxs = test_idxs,
    train_idxs = train_idxs,
    suffix = suffix
)
run_iq_regression(
    model_type = "5_pad",
    cgd = cgd,
    test_idxs = test_idxs,
    train_idxs = train_idxs,
    suffix = suffix
)
run_iq_regression(
    model_type = "3_pad",
    cgd = cgd,
    test_idxs = test_idxs,
    train_idxs = train_idxs,
    suffix = suffix
)

→ Computing sequence features

ℹ Seed: 60427

ℹ Training on 11531 intervals (82%) and testing on 2470 intervals (18%)

ℹ Saving the train and test indices to "/net/mraid20/ifs/wisdom/tanay_lab/tgdata/users/evghenic/Proj/polycomb/seq2epi_paper_raw/output/iq_5_prego_10_motifs_no_add_oriented_hg19_pcg_vs_txg"

ℹ Learning prego motifs (de-novo motifs)

ℹ Extracting sequences...

ℹ Inferring 5 prego motifs...

ℹ Using 15 bins of size 40 bp

ℹ Running regression for 5 motifs



── Running first regression ──



ℹ Using 15 bins of size 40 bp

ℹ Using "r2" as the final metric

ℹ Number of response variables: 1

ℹ Performing sampled optimization

ℹ Sampling 1 of the dataset

✔ Sampled 11530 rows from the data frame.

ℹ Sampling 0.1 of the dataset

✔ Sampled 1150 rows from the data frame.



── Generate candidate kmers 



── Regress each candidate kmer on sampled data 

ℹ Running regression on 10 candidate kmers

• Bidirectional: TRUE

• Spat bin size: 40

• Number of spatial bins 15

• Length 

ℹ Improvement in R2: 0.0137394961105447



── Running regression #4 ──



ℹ Using 15 bins of size 40 bp

ℹ Using "r2" as the final metric

ℹ Number of response variables: 1

ℹ Performing sampled optimization

ℹ Sampling 1 of the dataset

✔ Sampled 11530 rows from the data frame.

ℹ Sampling 0.1 of the dataset

✔ Sampled 1150 rows from the data frame.



── Generate candidate kmers 

ℹ Could not find any kmer when using a threshold of 0.08. Trying with 0.04



── Regress each candidate kmer on sampled data 

ℹ Running regression on 1 candidate kmers

• Bidirectional: TRUE

• Spat bin size: 40

• Number of spatial bins 15

• Length of sequence: 600

• Min gap: 0

• Max gap: 1

• Kmer length: 8

• Improve epsilon: 1e-04

• Min nuc prob: 0.001

• Uniform prior: 0.05

• Score metric: "r2"

• Seed: 60427

→ regressing with seed: "ACCAGA*CT"

→ "ACCAGA*CT", score (r2): 0.00447580469766442, validation score (r2): 0.00118388755109846

ℹ Performing regression on full data

ℹ Using 15 bins of siz

→ Matching JOLMA.KLF14_mono_DBD

→ Matched with "HOMER.Klf4", PSSM correlation = 0.813670564719117

→ Matching HOCOMOCO.ATF3_HUMAN.H11MO.0.A

→ Matched with "HOMER.CRE", PSSM correlation = 0.804690579718533

→ Matching HOMER.Elk4

→ Matched with "HOMER.ETS1", PSSM correlation = 0.883651252029852

→ Matching HOCOMOCO.KLF12_HUMAN.H11MO.0.C

→ Matched with "HOMER.Sp1", PSSM correlation = 0.91547374664879

→ Matching HOCOMOCO.E2F4_MOUSE.H11MO.0.A

→ Matched with "HOMER.E2F1", PSSM correlation = 0.78318579384703

Warning message in eval(family$initialize):
“non-integer #successes in a binomial glm!”
Warning message:
“glmnet.fit: algorithm did not converge”
ℹ Extracting sequences...

ℹ Computing motif energies for 2470 intervals and 14001 normalization intervals

ℹ Inferring the model on 2470 intervals



Number of motifs: 10

R^2 train: 0.204

R^2 test: 0.21



→ Saving the model to "/net/mraid20/ifs/wisdom/tanay_lab/tgdata/users/evghenic/Proj/polycomb/seq2epi_paper_raw/output/iq_5_prego_10

Warning message in eval(family$initialize):
“non-integer #successes in a binomial glm!”
Warning message:
“glmnet.fit: algorithm did not converge”
→ Removing "Klf4" changed the R^2 by -0.000829354663665738

Warning message in eval(family$initialize):
“non-integer #successes in a binomial glm!”
Warning message:
“glmnet.fit: algorithm did not converge”
→ Not removing "ETS" (changed the R^2 by only 0.00101081116260474).

Warning message in eval(family$initialize):
“non-integer #successes in a binomial glm!”
Warning message:
“glmnet.fit: algorithm did not converge”
→ Not removing "Sp1" (changed the R^2 by only 0.00533190922474011).

ℹ Removed 0 features with bits < 1.75

ℹ Removed 3 features with R^2 < 5e-04

Warning message in eval(family$initialize):
“non-integer #successes in a binomial glm!”
Warning message:
“glmnet.fit: algorithm did not converge”
✔ After filtering: Number of non-zero coefficients: 96 (out of 96). R^2: 0.201072472925478. Number of models: 7

ℹ Extracting sequences...



ℹ Using "r2" as the final metric

ℹ Number of response variables: 1

ℹ Motif is shorter than 15, extending with wildcards

ℹ Initializing regression with the following motif: "***GGGG*CGGG***"

ℹ Running regression

• Motif length: 15

• Bidirectional: TRUE

• Spat min: 0

• Spat max: 600

• Spat bin size: 40

• Number of bins: 15

• Improve epsilon: 1e-04

• Min nuc prob: 0.001

• Uniform prior: 0.05

• Score metric: "r2"

• Seed: 60427

✔ Finished running regression. Consensus: "CG"

✔ R^2: 0.1066

ℹ Best motif: "***GGGG*CGGG***", score (r2): 0.0990299054837721



── Running regression #2 ──



ℹ Using 15 bins of size 40 bp

ℹ Using "r2" as the final metric

ℹ Number of response variables: 1

ℹ Performing sampled optimization

ℹ Sampling 1 of the dataset

✔ Sampled 11530 rows from the data frame.

ℹ Sampling 0.1 of the dataset

✔ Sampled 1150 rows from the data frame.



── Generate candidate kmers 

ℹ Could not find any kmer when using a threshold of 0.08. Trying with 0.04



── Reg

...

Warning message in eval(family$initialize):
“non-integer #successes in a binomial glm!”
Warning message:
“glmnet.fit: algorithm did not converge”
→ R^2 added by GSC (-----GGGATTA---): -0.00141910864159378. Bits: 4.47255242712528

Warning message in eval(family$initialize):
“non-integer #successes in a binomial glm!”
Warning message:
“glmnet.fit: algorithm did not converge”
→ R^2 added by ETS1 (---GGAGGT-CC---): -0.000451958977188277. Bits: 2.73623909886248

Warning message in eval(family$initialize):
“non-integer #successes in a binomial glm!”
Warning message:
“glmnet.fit: algorithm did not converge”
→ R^2 added by JunD (--G-TGACGT-A---): 0.000471079437178695. Bits: 3.88172230410846

Warning message in eval(family$initialize):
“non-integer #successes in a binomial glm!”
Warning message:
“glmnet.fit: algorithm did not converge”
→ R^2 added by Klf4 (---CCCC-C------): -0.00112144001239949. Bits: 3.11156762179441

Warning message in eval(family$initialize):
“non-integer #successes in a bin

→ Computing sequence features

ℹ Seed: 60427

ℹ Training on 11531 intervals (82%) and testing on 2470 intervals (18%)

ℹ Saving the train and test indices to "/net/mraid20/ifs/wisdom/tanay_lab/tgdata/users/evghenic/Proj/polycomb/seq2epi_paper_raw/output/iq_3_pad_prego_10_motifs_no_add_oriented_hg19_pcg_vs_txg"

ℹ Learning prego motifs (de-novo motifs)

ℹ Extracting sequences...

ℹ Inferring 5 prego motifs...

ℹ Using 11 bins of size 40 bp

ℹ Running regression for 5 motifs



── Running first regression ──



ℹ Using 11 bins of size 40 bp

ℹ Using "r2" as the final metric

ℹ Number of response variables: 1

ℹ Performing sampled optimization

ℹ Sampling 1 of the dataset

✔ Sampled 11530 rows from the data frame.

ℹ Sampling 0.1 of the dataset

✔ Sampled 1150 rows from the data frame.



── Generate candidate kmers 

ℹ Could not find any kmer when using a threshold of 0.08. Trying with 0.04



── Regress each candidate kmer on sampled data 

ℹ Running regression on 1 candidate kmers

• B

✔ R^2: 1e-04

ℹ Best motif: "***CTTG*TCGC***", score (r2): 0.000258603616367314

ℹ R2 for models 1, 2, and 3: 0.0614267803405725

ℹ Improvement in R2: 5.95405789669545e-05



── Running regression #4 ──



ℹ Using 11 bins of size 40 bp

ℹ Using "r2" as the final metric

ℹ Number of response variables: 1

ℹ Performing sampled optimization

ℹ Sampling 1 of the dataset

✔ Sampled 11530 rows from the data frame.

ℹ Sampling 0.1 of the dataset

✔ Sampled 1150 rows from the data frame.



── Generate candidate kmers 

ℹ Could not find any kmer when using a threshold of 0.08. Trying with 0.04



── Regress each candidate kmer on sampled data 

ℹ Running regression on 1 candidate kmers

• Bidirectional: TRUE

• Spat bin size: 40

• Number of spatial bins 11

• Length of sequence: 440

• Min gap: 0

• Max gap: 1

• Kmer length: 8

• Improve epsilon: 1e-04

• Min nuc prob: 0.001

• Uniform prior: 0.05

• Score metric: "r2"

• Seed: 60427

→ regressing with seed: "CTTG*TCGC"

→ "CTTG*TCGC", score

→ Matched with "HOMER.Jun_AP1", PSSM correlation = 0.830632620028121

→ Matching JASPAR.TCF12

→ Matched with "HOMER.Klf4", PSSM correlation = 0.604851333435436

→ Matching HOCOMOCO.AP2B_HUMAN.H11MO.0.B

→ Matched with "HOMER.E2F4", PSSM correlation = 0.586716764830023

→ Matching JASPAR.KLF1

→ Matched with "HOMER.Klf4", PSSM correlation = 0.695964622605546

→ Matching JASPAR.YER184C

→ Matched with "HOMER.E2F4", PSSM correlation = 0.582727090829179

→ Matching JASPAR.CST6

→ Matched with "HOMER.JunD", PSSM correlation = 0.727047767419962

→ Matching JASPAR.ZBTB32

→ Matched with "HOMER.HIF_1a", PSSM correlation = 0.629812991884604

→ Matching JASPAR.ZNF768

→ Matched with "HOMER.GAGA_repeat", PSSM correlation = 0.627668059047814

Warning message in eval(family$initialize):
“non-integer #successes in a binomial glm!”
Warning message:
“glmnet.fit: algorithm did not converge”
ℹ Extracting sequences...

ℹ Computing motif energies for 2470 intervals and 14001 normalization intervals

ℹ In

Warning message:
“glmnet.fit: algorithm did not converge”
→ Removing "E2F4.2" changed the R^2 by -0.000652878072969154

Warning message in eval(family$initialize):
“non-integer #successes in a binomial glm!”
Warning message:
“glmnet.fit: algorithm did not converge”
→ Not removing "GAGA_repeat" (changed the R^2 by only 0.000735603015504266).

ℹ Removed 0 features with bits < 1.75

ℹ Removed 3 features with R^2 < 5e-04

Warning message in eval(family$initialize):
“non-integer #successes in a binomial glm!”
Warning message:
“glmnet.fit: algorithm did not converge”
✔ After filtering: Number of non-zero coefficients: 95 (out of 96). R^2: 0.168204873820012. Number of models: 7

ℹ Extracting sequences...

ℹ Computing motif energies for 2470 intervals and 14001 normalization intervals

ℹ Inferring the model on 2470 intervals



Number of motifs: 7

R^2 train: 0.168

R^2 test: 0.174



→ Saving the model to "/net/mraid20/ifs/wisdom/tanay_lab/tgdata/users/evghenic/Proj/polycomb/seq2epi_paper_raw

In [19]:
#new
suffix <- "_prego_10_motifs_no_add_oriented_hg19_pcg_vs_txg"
run_iq_regression(
    model_type = "5",
    cgd = cgd,
    test_idxs = test_idxs,
    train_idxs = train_idxs,
    suffix = suffix
)
run_iq_regression(
    model_type = "3",
    cgd = cgd,
    test_idxs = test_idxs,
    train_idxs = train_idxs,
    suffix = suffix
)
run_iq_regression(
    model_type = "5_pad",
    cgd = cgd,
    test_idxs = test_idxs,
    train_idxs = train_idxs,
    suffix = suffix
)
run_iq_regression(
    model_type = "3_pad",
    cgd = cgd,
    test_idxs = test_idxs,
    train_idxs = train_idxs,
    suffix = suffix
)

→ Computing sequence features

ℹ Seed: 60427

ℹ Training on 11531 intervals (82%) and testing on 2470 intervals (18%)

ℹ Saving the train and test indices to "/net/mraid20/ifs/wisdom/tanay_lab/tgdata/users/evghenic/Proj/polycomb/seq2epi_paper_raw/output/iq_5_prego_10_motifs_no_add_oriented_hg19_pcg_vs_txg"

ℹ Learning prego motifs (de-novo motifs)

ℹ Extracting sequences...

ℹ Inferring 5 prego motifs...

ℹ Using 15 bins of size 40 bp

ℹ Running regression for 5 motifs



── Running first regression ──



ℹ Using 15 bins of size 40 bp

ℹ Using "r2" as the final metric

ℹ Number of response variables: 1

ℹ Performing sampled optimization

ℹ Sampling 1 of the dataset

✔ Sampled 11530 rows from the data frame.

ℹ Sampling 0.1 of the dataset

✔ Sampled 1150 rows from the data frame.



── Generate candidate kmers 



── Regress each candidate kmer on sampled data 

ℹ Running regression on 10 candidate kmers

• Bidirectional: TRUE

• Spat bin size: 40

• Number of spatial bins 15

• Length 

ℹ Improvement in R2: 0.0137394961105447



── Running regression #4 ──



ℹ Using 15 bins of size 40 bp

ℹ Using "r2" as the final metric

ℹ Number of response variables: 1

ℹ Performing sampled optimization

ℹ Sampling 1 of the dataset

✔ Sampled 11530 rows from the data frame.

ℹ Sampling 0.1 of the dataset

✔ Sampled 1150 rows from the data frame.



── Generate candidate kmers 

ℹ Could not find any kmer when using a threshold of 0.08. Trying with 0.04



── Regress each candidate kmer on sampled data 

ℹ Running regression on 1 candidate kmers

• Bidirectional: TRUE

• Spat bin size: 40

• Number of spatial bins 15

• Length of sequence: 600

• Min gap: 0

• Max gap: 1

• Kmer length: 8

• Improve epsilon: 1e-04

• Min nuc prob: 0.001

• Uniform prior: 0.05

• Score metric: "r2"

• Seed: 60427

→ regressing with seed: "ACCAGA*CT"

→ "ACCAGA*CT", score (r2): 0.00447580469766431, validation score (r2): 0.00118388755109846

ℹ Performing regression on full data

ℹ Using 15 bins of siz

→ Matching JOLMA.KLF14_mono_DBD

→ Matched with "HOMER.Klf4", PSSM correlation = 0.813670564719117

→ Matching HOCOMOCO.ATF3_HUMAN.H11MO.0.A

→ Matched with "HOMER.CRE", PSSM correlation = 0.804690579718533

→ Matching HOMER.Elk4

→ Matched with "HOMER.ETS1", PSSM correlation = 0.883651252029852

→ Matching HOCOMOCO.KLF12_HUMAN.H11MO.0.C

→ Matched with "HOMER.Sp1", PSSM correlation = 0.91547374664879

→ Matching HOCOMOCO.E2F4_MOUSE.H11MO.0.A

→ Matched with "HOMER.E2F1", PSSM correlation = 0.78318579384703

Warning message in eval(family$initialize):
“non-integer #successes in a binomial glm!”
Warning message:
“glmnet.fit: algorithm did not converge”
ℹ Extracting sequences...

ℹ Computing motif energies for 2470 intervals and 14001 normalization intervals

ℹ Inferring the model on 2470 intervals



Number of motifs: 10

R^2 train: 0.204

R^2 test: 0.21



→ Saving the model to "/net/mraid20/ifs/wisdom/tanay_lab/tgdata/users/evghenic/Proj/polycomb/seq2epi_paper_raw/output/iq_5_prego_10

Warning message in eval(family$initialize):
“non-integer #successes in a binomial glm!”
Warning message:
“glmnet.fit: algorithm did not converge”
→ Removing "Klf4" changed the R^2 by -0.000829354663665682

Warning message in eval(family$initialize):
“non-integer #successes in a binomial glm!”
Warning message:
“glmnet.fit: algorithm did not converge”
→ Not removing "ETS" (changed the R^2 by only 0.00101081116260485).

Warning message in eval(family$initialize):
“non-integer #successes in a binomial glm!”
Warning message:
“glmnet.fit: algorithm did not converge”
→ Not removing "Sp1" (changed the R^2 by only 0.00533190922474014).

ℹ Removed 0 features with bits < 1.75

ℹ Removed 3 features with R^2 < 5e-04

Warning message in eval(family$initialize):
“non-integer #successes in a binomial glm!”
Warning message:
“glmnet.fit: algorithm did not converge”
✔ After filtering: Number of non-zero coefficients: 96 (out of 96). R^2: 0.201072472925478. Number of models: 7

ℹ Extracting sequences...



ℹ Using "r2" as the final metric

ℹ Number of response variables: 1

ℹ Motif is shorter than 15, extending with wildcards

ℹ Initializing regression with the following motif: "***GGGG*CGGG***"

ℹ Running regression

• Motif length: 15

• Bidirectional: TRUE

• Spat min: 0

• Spat max: 600

• Spat bin size: 40

• Number of bins: 15

• Improve epsilon: 1e-04

• Min nuc prob: 0.001

• Uniform prior: 0.05

• Score metric: "r2"

• Seed: 60427

✔ Finished running regression. Consensus: "CG"

✔ R^2: 0.1066

ℹ Best motif: "***GGGG*CGGG***", score (r2): 0.0990299054837721



── Running regression #2 ──



ℹ Using 15 bins of size 40 bp

ℹ Using "r2" as the final metric

ℹ Number of response variables: 1

ℹ Performing sampled optimization

ℹ Sampling 1 of the dataset

✔ Sampled 11530 rows from the data frame.

ℹ Sampling 0.1 of the dataset

✔ Sampled 1150 rows from the data frame.



── Generate candidate kmers 

ℹ Could not find any kmer when using a threshold of 0.08. Trying with 0.04



── Reg

...

Warning message in eval(family$initialize):
“non-integer #successes in a binomial glm!”
Warning message:
“glmnet.fit: algorithm did not converge”
→ R^2 added by GSC (-----GGGATTA---): -0.00141910864159391. Bits: 4.47255242712528

Warning message in eval(family$initialize):
“non-integer #successes in a binomial glm!”
Warning message:
“glmnet.fit: algorithm did not converge”
→ R^2 added by ETS1 (---GGAGGT-CC---): -0.000451958977188485. Bits: 2.73623909886248

Warning message in eval(family$initialize):
“non-integer #successes in a binomial glm!”
Warning message:
“glmnet.fit: algorithm did not converge”
→ R^2 added by JunD (--G-TGACGT-A---): 0.000471079437178668. Bits: 3.88172230410846

Warning message in eval(family$initialize):
“non-integer #successes in a binomial glm!”
Warning message:
“glmnet.fit: algorithm did not converge”
→ R^2 added by Klf4 (---CCCC-C------): -0.00112144001239957. Bits: 3.11156762179441

Warning message in eval(family$initialize):
“non-integer #successes in a bin

→ Computing sequence features

ℹ Seed: 60427

ℹ Training on 11531 intervals (82%) and testing on 2470 intervals (18%)

ℹ Saving the train and test indices to "/net/mraid20/ifs/wisdom/tanay_lab/tgdata/users/evghenic/Proj/polycomb/seq2epi_paper_raw/output/iq_3_pad_prego_10_motifs_no_add_oriented_hg19_pcg_vs_txg"

ℹ Learning prego motifs (de-novo motifs)

ℹ Extracting sequences...

ℹ Inferring 5 prego motifs...

ℹ Using 11 bins of size 40 bp

ℹ Running regression for 5 motifs



── Running first regression ──



ℹ Using 11 bins of size 40 bp

ℹ Using "r2" as the final metric

ℹ Number of response variables: 1

ℹ Performing sampled optimization

ℹ Sampling 1 of the dataset

✔ Sampled 11530 rows from the data frame.

ℹ Sampling 0.1 of the dataset

✔ Sampled 1150 rows from the data frame.



── Generate candidate kmers 

ℹ Could not find any kmer when using a threshold of 0.08. Trying with 0.04



── Regress each candidate kmer on sampled data 

ℹ Running regression on 1 candidate kmers

• B

✔ R^2: 1e-04

ℹ Best motif: "***CTTG*TCGC***", score (r2): 0.000258603616367315

ℹ R2 for models 1, 2, and 3: 0.0614267803405725

ℹ Improvement in R2: 5.95405789669545e-05



── Running regression #4 ──



ℹ Using 11 bins of size 40 bp

ℹ Using "r2" as the final metric

ℹ Number of response variables: 1

ℹ Performing sampled optimization

ℹ Sampling 1 of the dataset

✔ Sampled 11530 rows from the data frame.

ℹ Sampling 0.1 of the dataset

✔ Sampled 1150 rows from the data frame.



── Generate candidate kmers 

ℹ Could not find any kmer when using a threshold of 0.08. Trying with 0.04



── Regress each candidate kmer on sampled data 

ℹ Running regression on 1 candidate kmers

• Bidirectional: TRUE

• Spat bin size: 40

• Number of spatial bins 11

• Length of sequence: 440

• Min gap: 0

• Max gap: 1

• Kmer length: 8

• Improve epsilon: 1e-04

• Min nuc prob: 0.001

• Uniform prior: 0.05

• Score metric: "r2"

• Seed: 60427

→ regressing with seed: "CTTG*TCGC"

→ "CTTG*TCGC", score

→ Matched with "HOMER.Jun_AP1", PSSM correlation = 0.830632620028121

→ Matching JASPAR.TCF12

→ Matched with "HOMER.Klf4", PSSM correlation = 0.604851333435436

→ Matching HOCOMOCO.AP2B_HUMAN.H11MO.0.B

→ Matched with "HOMER.E2F4", PSSM correlation = 0.586716764830023

→ Matching JASPAR.KLF1

→ Matched with "HOMER.Klf4", PSSM correlation = 0.695964622605546

→ Matching JASPAR.YER184C

→ Matched with "HOMER.E2F4", PSSM correlation = 0.582727090829179

→ Matching JASPAR.CST6

→ Matched with "HOMER.JunD", PSSM correlation = 0.727047767419962

→ Matching JASPAR.ZBTB32

→ Matched with "HOMER.HIF_1a", PSSM correlation = 0.629812991884604

→ Matching JASPAR.ZNF768

→ Matched with "HOMER.GAGA_repeat", PSSM correlation = 0.627668059047814

Warning message in eval(family$initialize):
“non-integer #successes in a binomial glm!”
Warning message:
“glmnet.fit: algorithm did not converge”
ℹ Extracting sequences...

ℹ Computing motif energies for 2470 intervals and 14001 normalization intervals

ℹ In

Warning message:
“glmnet.fit: algorithm did not converge”
→ Removing "E2F4.2" changed the R^2 by -0.000652878072969043

Warning message in eval(family$initialize):
“non-integer #successes in a binomial glm!”
Warning message:
“glmnet.fit: algorithm did not converge”
→ Not removing "GAGA_repeat" (changed the R^2 by only 0.000735603015504224).

ℹ Removed 0 features with bits < 1.75

ℹ Removed 3 features with R^2 < 5e-04

Warning message in eval(family$initialize):
“non-integer #successes in a binomial glm!”
Warning message:
“glmnet.fit: algorithm did not converge”
✔ After filtering: Number of non-zero coefficients: 95 (out of 96). R^2: 0.168204873820012. Number of models: 7

ℹ Extracting sequences...

ℹ Computing motif energies for 2470 intervals and 14001 normalization intervals

ℹ Inferring the model on 2470 intervals



Number of motifs: 7

R^2 train: 0.168

R^2 test: 0.174



→ Saving the model to "/net/mraid20/ifs/wisdom/tanay_lab/tgdata/users/evghenic/Proj/polycomb/seq2epi_paper_raw

In [9]:
suffix <- "_prego_10_motifs_no_add_oriented_hg19_pcg_vs_txg"
run_iq_regression(
    model_type = "5",
    cgd = cgd,
    test_idxs = test_idxs,
    train_idxs = train_idxs,
    suffix = suffix
)
run_iq_regression(
    model_type = "3",
    cgd = cgd,
    test_idxs = test_idxs,
    train_idxs = train_idxs,
    suffix = suffix
)
run_iq_regression(
    model_type = "5_pad",
    cgd = cgd,
    test_idxs = test_idxs,
    train_idxs = train_idxs,
    suffix = suffix
)
run_iq_regression(
    model_type = "3_pad",
    cgd = cgd,
    test_idxs = test_idxs,
    train_idxs = train_idxs,
    suffix = suffix
)

→ Computing sequence features

ℹ Seed: 60427

ℹ Training on 6354 intervals (81%) and testing on 1460 intervals (19%)

ℹ Saving the train and test indices to "/net/mraid20/ifs/wisdom/tanay_lab/tgdata/users/evghenic/Proj/polycomb/seq2epi_paper/output/iq_5_prego_10_motifs_no_add_oriented_hg19_pcg_vs_txg"

ℹ Learning prego motifs (de-novo motifs)

ℹ Extracting sequences...

ℹ Inferring 5 prego motifs...

ℹ Using 15 bins of size 40 bp

ℹ Running regression for 5 motifs



── Running first regression ──



ℹ Using 15 bins of size 40 bp

ℹ Using "ks" as the final metric

ℹ Number of response variables: 1

ℹ Performing sampled optimization

ℹ Sampling 1 of the dataset

ℹ Stratified sampling

ℹ sampled 1973 zeros and 4381 ones

ℹ Sampling 0.1 of the dataset

ℹ Stratified sampling

ℹ sampled 197 zeros and 438 ones



── Generate candidate kmers 



── Regress each candidate kmer on sampled data 

ℹ Running regression on 9 candidate kmers

• Bidirectional: TRUE

• Spat bin size: 40

• Number of s

R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call


ℹ Performing regression on full data

ℹ Using 15 bins of size 40 bp

ℹ Using "r2" as the final metric

ℹ Number of response variables: 1

ℹ Motif is shorter than 15, extending with wildcards

ℹ Initializing regression with the following motif: "***CGCA*GCGC***"

ℹ Running regression

• Motif length: 15

• Bidirectional: TRUE

• Spat min: 0

• Spat max: 600

• Spat bin size: 40

• Number of bins: 15

• Improve epsilon: 0.0001

• Min nuc prob: 0.001

• Uniform prior: 0.05

• Score metric: "r2"

• Seed: 60427

✔ Finished running regression. Consensus: "GMGCA*GCGC"

✔ R^2: 0.0143

ℹ Best motif: "***CGCA*GCGC***", score (r2): 0.0159218532385309

ℹ KS statistic: 0.125410688670482

ℹ KS test statistic for models 1 and 2: 0.440336924652635

ℹ Improvement in KS test statistic: 0.0204282580876991



── Running regression #3 ──



ℹ Using 15 bins of size 40 bp

ℹ Using "r2" as the final metric

ℹ Number of response variables: 1

ℹ Performing sampled optimization

ℹ Sampling 1 of the dataset

✔ Sa

R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call


ℹ Infering energies...

ℹ Running final regression, number of features: 108

✔ Finished running model. Number of non-zero coefficients: 108 (out of 108). R^2: 0.33762130834668

→ Matching SCENIC.cisbp__M02599

→ Matched with "HOMER.Elk4", PSSM correlation = 0.717623902313311

→ Matching JASPAR.Fli-Erg-c

→ Matched with "HOMER.ETS1", PSSM correlation = 0.885306330928511

→ Matching SCENIC.transfac_public__M00039

→ Matched with "HOMER.Atf7", PSSM correlation = 0.618162092565979

→ Matching SCENIC.flyfactorsurvey__phol_SANGER_5_FBgn0035997

→ Matched with "HOMER.YY1", PSSM correlation = 0.80340613013752

→ Matching SCENIC.homer__GGCCCCGCCCCC_Sp1

→ Matched with "HOMER.KLF5", PSSM correlation = 0.800044715138123

→ Matching SCENIC.elemento__TTCCGCC

→ Matched with "HOMER.ETS", PSSM correlation = 0.617312971896137

→ Matching e2

→ Matched with "HOMER.NRF1", PSSM correlation = 0.883282215680623

→ Matching SCENIC.dbtfbs__ETV5_HepG2_ENCSR096IIB_merged_N1

→ Matched with "HOMER.Elk1", PSSM c

...

ℹ Infering energies...

ℹ Running final regression, number of features: 108

✔ Finished running model. Number of non-zero coefficients: 107 (out of 108). R^2: 0.258395854254954

→ Matching SCENIC.cisbp__M01050

→ Matched with "HOMER.TCFL2", PSSM correlation = 0.52739306175118

→ Matching SCENIC.homer__GGGGGGGG_Maz

→ Matched with "HOMER.Egr1", PSSM correlation = 0.600683575607317

→ Matching SCENIC.cisbp__M02363

→ Matched with "HOMER.Maz", PSSM correlation = 0.684830270210646

→ Matching SCENIC.cisbp__M00596

→ Matched with "HOMER.ZNF711", PSSM correlation = 0.530874662525828

→ Matching SCENIC.yetfasco__YGL181W_694

→ Matched with "HOMER.CRE", PSSM correlation = 0.459651703958793

→ Matching HOMER.NRF1

→ Matched with "HOMER.NRF1", PSSM correlation = 0.594761840360188

→ Matching SCENIC.cisbp__M01350

→ Matched with "HOMER.MafA", PSSM correlation = 0.587102641026766

→ Matching SCENIC.predrem__nrMotif1968

→ Matched with "HOMER.Ascl1", PSSM correlation = 0.583594149787116

→ Matching

R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call


Warning message in ks.test.default(plot_df$observed[plot_df$energy == "0-3"], plot_df$observed[plot_df$energy == :
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(plot_df$observed[plot_df$energy == "0-3"], plot_df$observed[plot_df$energy == :
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(plot_df$observed[plot_df$energy == "0-3"], plot_df$observed[plot_df$energy == :
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(plot_df$observed[plot_df$energy == "0-3"], plot_df$observed[plot_df$energy == :
“p-value will be approximate in the presence of ties”
ℹ Saving plot...

Warning message:
“Removed 46 rows containing missing values or values outside the scale range
(`geom_line()`).”
Warning message:
“Removed 46 rows containing missing values or values outside the scale range
(`geom_line()`).”
Warning message:
“Removed 46 rows containing missing values or values outside the 

R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call


Warning message in ks.test.default(plot_df$observed[plot_df$energy == "0-3"], plot_df$observed[plot_df$energy == :
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(plot_df$observed[plot_df$energy == "0-3"], plot_df$observed[plot_df$energy == :
“p-value will be approximate in the presence of ties”
ℹ Saving plot...

Warning message:
“Removed 46 rows containing missing values or values outside the scale range
(`geom_line()`).”
Warning message:
“Removed 46 rows containing missing values or values outside the scale range
(`geom_line()`).”
Warning message:
“Removed 46 rows containing missing values or values outside the scale range
(`geom_line()`).”
Warning message:
“Removed 46 rows containing missing values or values outside the scale range
(`geom_line()`).”
Warning message:
“Removed 46 rows containing missing values or values outside the scale range
(`geom_line()`).”
Warning message:
“Removed 32 rows containing missing values or values outside the sc